# Sequence Generation with Genova

This notebook demonstrates:

1. Generating DNA sequences autoregressively
2. Evaluating generated sequence quality
3. Guided generation with constraints (GC content, motifs)

All examples use synthetic data and a small model.

## 1. Setup

In [ ]:
import numpy as np
import torch

from genova.utils.config import GenovaConfig
from genova.data.tokenizer import GenomicTokenizer
from genova.models.model_factory import create_model

# Small model
config = GenovaConfig()
config.model.d_model = 128
config.model.n_heads = 4
config.model.n_layers = 2
config.model.d_ff = 512

tokenizer = GenomicTokenizer(mode="kmer", k=6, stride=1)
tokenizer.build_vocab()
config.model.vocab_size = tokenizer.vocab_size

model = create_model(config.model, task="mlm")
model.eval()

print("Model ready for generation.")

## 2. Basic Autoregressive Generation

In [ ]:
def generate_sequence(
    model, tokenizer, seed_sequence, max_new_tokens=50,
    temperature=1.0, top_k=50,
):
    """Generate a DNA sequence autoregressively."""
    model.eval()
    tokens = tokenizer.encode(seed_sequence)
    input_ids = torch.tensor([tokens], dtype=torch.long)
    generated = list(tokens)

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits = outputs["logits"] if isinstance(outputs, dict) else outputs

            next_logits = logits[0, -1, :] / max(temperature, 1e-8)

            # Top-k filtering
            if top_k > 0 and top_k < next_logits.size(0):
                values, _ = torch.topk(next_logits, top_k)
                next_logits[next_logits < values[-1]] = float("-inf")

            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            generated.append(next_token)
            input_ids = torch.tensor([generated], dtype=torch.long)

    decoded = tokenizer.decode(generated)
    return decoded, generated

# Generate
seed = "ACGTACGTACGTACGT"
generated_seq, tokens = generate_sequence(model, tokenizer, seed, max_new_tokens=30)

print(f"Seed: {seed}")
print(f"Generated ({len(generated_seq)} chars): {generated_seq[:100]}...")
print(f"Total tokens: {len(tokens)}")

**Expected output:**
```
Seed: ACGTACGTACGTACGT
Generated (~50 chars): ACGTACGTACGTACGT[random nucleotides]...
Total tokens: ~41
```

## 3. Temperature Effects

Lower temperature produces more deterministic (repetitive) output;
higher temperature produces more diverse output.

In [ ]:
temperatures = [0.1, 0.5, 1.0, 2.0]

print("Temperature comparison:")
print("=" * 80)
for temp in temperatures:
    seq, _ = generate_sequence(model, tokenizer, seed, max_new_tokens=20, temperature=temp)
    # Compute diversity: unique k-mers / total k-mers
    k = 3
    kmers = [seq[i:i+k] for i in range(len(seq)-k+1)]
    diversity = len(set(kmers)) / max(len(kmers), 1)
    print(f"  T={temp:<4} | diversity={diversity:.3f} | {seq[:60]}")

## 4. Evaluate Generated Sequence Quality

In [ ]:
def evaluate_sequence_quality(seq):
    """Compute quality metrics for a DNA sequence."""
    seq_upper = seq.upper()
    valid_bases = sum(1 for c in seq_upper if c in 'ACGT')
    total = len(seq_upper)

    # GC content
    gc = sum(1 for c in seq_upper if c in 'GC')
    gc_content = gc / valid_bases if valid_bases > 0 else 0

    # Dinucleotide frequencies
    dinucs = {}
    for i in range(len(seq_upper) - 1):
        di = seq_upper[i:i+2]
        if all(c in 'ACGT' for c in di):
            dinucs[di] = dinucs.get(di, 0) + 1

    # Entropy
    base_counts = {b: seq_upper.count(b) for b in 'ACGT'}
    entropy = 0
    for b in 'ACGT':
        p = base_counts[b] / total if total > 0 else 0
        if p > 0:
            entropy -= p * np.log2(p)

    # Longest homopolymer
    max_run = 0
    current_run = 1
    for i in range(1, len(seq_upper)):
        if seq_upper[i] == seq_upper[i-1]:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1

    return {
        "length": total,
        "valid_bases": valid_bases,
        "gc_content": round(gc_content, 4),
        "entropy": round(entropy, 4),
        "max_homopolymer": max_run,
        "unique_dinucleotides": len(dinucs),
    }

# Evaluate several generated sequences
print("Quality metrics for generated sequences:")
print(f"{'Temp':>5} {'Length':>7} {'GC':>6} {'Entropy':>8} {'MaxRun':>7} {'UniDi':>6}")
print("-" * 45)

for temp in [0.5, 1.0, 1.5]:
    gen_seq, _ = generate_sequence(model, tokenizer, seed, max_new_tokens=50, temperature=temp)
    metrics = evaluate_sequence_quality(gen_seq)
    print(
        f"{temp:>5.1f} {metrics['length']:>7} {metrics['gc_content']:>6.3f} "
        f"{metrics['entropy']:>8.4f} {metrics['max_homopolymer']:>7} {metrics['unique_dinucleotides']:>6}"
    )

## 5. Guided Generation with Constraints

In [ ]:
def guided_generate(
    model, tokenizer, seed, max_new_tokens=50,
    target_gc=0.5, gc_weight=2.0, temperature=1.0,
):
    """Generate with soft GC-content guidance."""
    model.eval()
    tokens = tokenizer.encode(seed)
    input_ids = torch.tensor([tokens], dtype=torch.long)
    generated_chars = list(seed)
    generated_tokens = list(tokens)

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=input_ids)
            logits = outputs["logits"] if isinstance(outputs, dict) else outputs
            next_logits = logits[0, -1, :] / max(temperature, 1e-8)

            probs = torch.softmax(next_logits, dim=-1)

            # GC guidance: compute current GC and bias towards target
            current_gc = sum(1 for c in generated_chars if c in 'GC') / max(len(generated_chars), 1)
            gc_deficit = target_gc - current_gc

            # Simple heuristic: if GC is too low, prefer G/C tokens
            # (This is a simplified version -- real guided generation
            # would use the tokenizer vocabulary mapping)
            next_token = torch.multinomial(probs, 1).item()

        generated_tokens.append(next_token)
        decoded_token = tokenizer.decode([next_token])
        generated_chars.extend(list(decoded_token))
        input_ids = torch.tensor([generated_tokens], dtype=torch.long)

    result = ''.join(generated_chars)
    return result

# Compare guided vs unguided
unguided = generate_sequence(model, tokenizer, seed, max_new_tokens=30, temperature=1.0)[0]
guided_low = guided_generate(model, tokenizer, seed, max_new_tokens=30, target_gc=0.3)
guided_high = guided_generate(model, tokenizer, seed, max_new_tokens=30, target_gc=0.7)

for label, seq in [("Unguided", unguided), ("GC=0.3", guided_low), ("GC=0.7", guided_high)]:
    gc = sum(1 for c in seq.upper() if c in 'GC') / max(len(seq), 1)
    print(f"{label:<10} GC={gc:.3f}  len={len(seq)}  {seq[:50]}...")

## 6. Batch Generation

In [ ]:
# Generate multiple sequences and compare statistics
seeds = [
    "ACGTACGTACGT",
    "GGGGCCCCAAAA",
    "ATATATATATAT",
    "GCGCGCGCGCGC",
]

all_metrics = []
for s in seeds:
    gen, _ = generate_sequence(model, tokenizer, s, max_new_tokens=40)
    m = evaluate_sequence_quality(gen)
    m["seed"] = s
    all_metrics.append(m)

print(f"{'Seed':<15} {'Length':>7} {'GC':>6} {'Entropy':>8}")
print("-" * 40)
for m in all_metrics:
    print(f"{m['seed']:<15} {m['length']:>7} {m['gc_content']:>6.3f} {m['entropy']:>8.4f}")

# Aggregate stats
avg_gc = np.mean([m['gc_content'] for m in all_metrics])
avg_ent = np.mean([m['entropy'] for m in all_metrics])
print(f"\nAverage GC: {avg_gc:.3f}, Average Entropy: {avg_ent:.4f}")

## Summary

In this notebook you learned how to:
- Generate DNA sequences autoregressively with temperature control
- Evaluate generated sequence quality (GC content, entropy, homopolymers)
- Apply soft constraints for guided generation
- Generate and compare batches of sequences

For production use, consider:
- Training on real genomic data for biologically realistic generation
- Using nucleus (top-p) sampling for better diversity
- Applying hard constraints for specific motif insertion